# Workshop de Análise de Dados - Dia 1 (SOLUÇÃO DO INSTRUTOR)
## Fundamentos e Data Cleaning

**Este notebook contém todas as respostas e comentários do instrutor. NÃO distribua antes do workshop.**

In [ ]:
import os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q pandas numpy matplotlib seaborn
    !mkdir -p data/prepared
    BASE_URL = 'https://github.com/caio-gomes/workshop-analise-dados/releases/download/v1/data'
    for f in ['orders.csv', 'items.csv', 'reviews.csv']:
        !wget -q -O data/prepared/{f} {BASE_URL}/{f}
    DATA_DIR = 'data/prepared'
    print('Ambiente Colab configurado!')
else:
    DATA_DIR = '../data/prepared'

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')
pd.set_option('display.max_columns', 50)

orders = pd.read_csv(f'{DATA_DIR}/orders.csv')
items = pd.read_csv(f'{DATA_DIR}/items.csv')
reviews = pd.read_csv(f'{DATA_DIR}/reviews.csv')

print(f'Orders:  {orders.shape[0]:,} linhas, {orders.shape[1]} colunas')
print(f'Items:   {items.shape[0]:,} linhas, {items.shape[1]} colunas')
print(f'Reviews: {reviews.shape[0]:,} linhas, {reviews.shape[1]} colunas')

---
## Bloco 1: Diagnóstico

In [ ]:
# Estrutura
for name, df in [('orders', orders), ('items', items), ('reviews', reviews)]:
    print(f'\n=== {name.upper()} ===')
    print(f'Shape: {df.shape}')
    print(df.dtypes)
    display(df.head(3))

In [ ]:
# Missings
for name, df in [('orders', orders), ('items', items), ('reviews', reviews)]:
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    if len(missing) > 0:
        print(f'\n{name.upper()} - Valores ausentes:')
        for col, count in missing.items():
            print(f'  {col}: {count} ({count/len(df)*100:.1f}%)')

In [ ]:
# DECISÃO INSTRUTOR: Investigar se missings em review_score são aleatórios
# Resultado: NÃO são. Pedidos de alto valor e categorias tech têm mais missings.
# Isso é Missing At Random (MAR) - o padrão de ausência depende de outra variável observada.

reviews_items = reviews.merge(items, on='order_id', how='left')

print('Preço médio - COM score:', reviews_items[reviews_items['review_score'].notna()]['price'].mean().round(2))
print('Preço médio - SEM score:', reviews_items[reviews_items['review_score'].isna()]['price'].mean().round(2))

# Por categoria
miss_rate = reviews_items.groupby('category')['review_score'].apply(lambda x: x.isna().mean()).sort_values(ascending=False)
print('\nTop 10 categorias com mais missings:')
print(miss_rate.head(10).round(3))

In [ ]:
# Duplicatas
print(f'Duplicatas exatas em orders: {orders.duplicated().sum()}')
print(f'Order_ids duplicados: {orders["order_id"].duplicated().sum()}')

dup_ids = orders[orders['order_id'].duplicated(keep=False)]['order_id'].unique()
print(f'Pedidos com mais de um registro: {len(dup_ids)}')

# DECISÃO INSTRUTOR: São duplicatas parciais - mesmo pedido com timestamps ligeiramente diferentes.
# Provavelmente re-processamento no sistema. Vamos manter o último registro.
sample_id = dup_ids[0]
display(orders[orders['order_id'] == sample_id])

In [ ]:
# Describe
display(items.describe())

# DECISÃO INSTRUTOR: Preço máximo de R$6.735 parece alto mas não absurdo.
# Mas olhando P99 vs max, há salto grande. Investigar outliers.
print('\nPercentis de preço:')
for p in [90, 95, 99, 99.5, 100]:
    print(f'  P{p}: R${items["price"].quantile(p/100):,.2f}')

In [ ]:
# Categorias
print(f'Total de categorias únicas: {items["category"].nunique()}')
print(f'\nTop 20 categorias:')
print(items['category'].value_counts().head(20))

# DECISÃO INSTRUTOR: Muitas categorias (~96) com variações de grafia.
# Ex: health_beauty, Health_Beauty, heath_beauty -> mesmo produto.
# Precisamos padronizar.

**Problemas identificados no diagnóstico:**

1. Missings não-aleatórios em review_score (~6%), correlacionados com preço alto e categorias tech
2. ~500 duplicatas parciais em orders (mesmo order_id, timestamps diferentes)
3. Formatos de data misturados em purchase_date (ISO e BR dd/mm/yyyy)
4. Outliers em price (valores muito altos, possivelmente compras B2B)
5. Categorias com grafia inconsistente (maiúsculas, espaços, typos)

---
## Bloco 2A: Limpeza Estrutural

In [ ]:
# 2.1 Duplicatas: manter último registro por order_id
print(f'Orders antes: {len(orders)}')
orders_clean = orders.sort_values('purchase_date').drop_duplicates(subset='order_id', keep='last')
print(f'Orders depois: {len(orders_clean)}')
print(f'Removidos: {len(orders) - len(orders_clean)}')

In [ ]:
# 2.2 Datas: converter com format='mixed' e dayfirst=True
date_cols = ['purchase_date', 'approved_date', 'delivered_date', 'estimated_delivery_date']
for col in date_cols:
    orders_clean[col] = pd.to_datetime(orders_clean[col], format='mixed', dayfirst=True)
    print(f'{col}: {orders_clean[col].dtype}, nulls: {orders_clean[col].isna().sum()}')

print(f'\nPeriodo: {orders_clean["purchase_date"].min()} a {orders_clean["purchase_date"].max()}')

### Mini-checkpoint: Limpeza Estrutural

In [ ]:
# MINI-CHECKPOINT
checks = {
    'Duplicatas removidas': orders_clean['order_id'].duplicated().sum() == 0,
    'Datas convertidas': orders_clean['purchase_date'].dtype == 'datetime64[ns]',
}
for check, passed in checks.items():
    status = 'OK' if passed else 'VERIFICAR'
    print(f'  [{status}] {check}')

---
## Bloco 2B: Limpeza Semântica

In [ ]:
# 2.3 Missings: manter NaN em review_score
# DECISÃO: Como o missing NÃO é aleatório (MAR), imputar distorceria a análise.
# Vamos manter NaN e filtrar nas análises que usam score.
reviews_clean = reviews.copy()
print(f'Reviews scores válidos: {reviews_clean["review_score"].notna().sum()}')
print(f'Reviews scores missing: {reviews_clean["review_score"].isna().sum()}')

In [ ]:
# 2.4 Categorias: padronizar e corrigir typos
items_clean = items.copy()
items_clean['category'] = (
    items_clean['category']
    .str.lower()
    .str.strip()
    .str.replace(' ', '_', regex=False)
)

# Categorias suspeitas (poucos itens = possível typo)
cats = items_clean['category'].value_counts()
print('Categorias com poucos itens (possíveis typos):')
print(cats[cats < 50])

# Dicionário de correção completo
typo_fixes = {
    'heath_beauty': 'health_beauty',
    'health_beuty': 'health_beauty',
    'sport_leisure': 'sports_leisure',
    'house_wares': 'housewares',
    'housware': 'housewares',
    'computer_accessories': 'computers_accessories',
    'furnitur_decor': 'furniture_decor',
    'watches_gift': 'watches_gifts',
    'telefony': 'telephony',
    'telephoni': 'telephony',
    'bed_bath': 'bed_bath_table',
}
items_clean['category'] = items_clean['category'].replace(typo_fixes)
print(f'\nAntes: {items["category"].nunique()} categorias')
print(f'Depois: {items_clean["category"].nunique()} categorias')

In [ ]:
# 2.5 Outliers: flag ao invés de remoção
threshold = items_clean['price'].quantile(0.99)
items_clean['is_high_value'] = items_clean['price'] > threshold

print(f'Threshold (P99): R${threshold:,.2f}')
print(f'Itens normais: {(~items_clean["is_high_value"]).sum():,}')
print(f'Itens alto valor: {items_clean["is_high_value"].sum():,}')

In [ ]:
# Desafio Bloco 2: delivery_delay_days
orders_clean['delivery_delay_days'] = (orders_clean['delivered_date'] - orders_clean['estimated_delivery_date']).dt.days
print(f'Mediana do atraso: {orders_clean["delivery_delay_days"].median():.0f} dias')
print(f'Média do atraso: {orders_clean["delivery_delay_days"].mean():.1f} dias')
print(f'% atrasados (>0): {(orders_clean["delivery_delay_days"] > 0).mean()*100:.1f}%')

---
## Bloco 3: Exploração (30 min)

In [ ]:
# Distribuições
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

reviews_clean['review_score'].dropna().value_counts().sort_index().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Distribuição de Notas')
axes[0].set_xlabel('Nota')

items_clean[~items_clean['is_high_value']]['price'].hist(bins=50, ax=axes[1], color='coral')
axes[1].set_title('Distribuição de Preços (excl. alto valor)')
axes[1].set_xlabel('Preço (R$)')

plt.tight_layout()
plt.show()

In [ ]:
# Volume ao longo do tempo
orders_clean.set_index('purchase_date').resample('ME')['order_id'].count().plot(figsize=(14, 5), marker='o', color='steelblue')
plt.title('Volume de Pedidos por Mês')
plt.ylabel('Quantidade')
plt.tight_layout()
plt.show()

In [ ]:
# Top categorias
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

items_clean['category'].value_counts().head(15).plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Top 15 por Volume')

items_clean.groupby('category')['price'].sum().nlargest(15).plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_title('Top 15 por Receita')

plt.tight_layout()
plt.show()

**Observações da exploração:**

1. Distribuição de notas é bimodal: concentração em 5 e 1. Média pode ser enganosa.
2. Volume cresceu até ~nov/2017, depois se estabilizou com sazonalidade mensal.
3. Categorias top em volume e receita não são idênticas (ex: cama_mesa_banho lidera volume mas não receita).

---
## Bloco 4: Perguntas e Preparação

### Perguntas Analíticas (Solução)

**Pergunta 1:** Em quais categorias a insatisfação está concentrada, e qual o peso dessas categorias na receita total?
- Dados: items + reviews (merge por order_id)
- Métrica: nota média por categoria + % da receita
- Impacto: permite priorizar categorias para melhoria

**Pergunta 2:** Qual o impacto quantitativo do atraso na entrega sobre a avaliação do cliente?
- Dados: orders + reviews (merge por order_id)
- Métrica: nota média por faixa de atraso (delivered - estimated)
- Impacto: justifica investimento em logística se a relação for forte

**Pergunta 3:** Existe sazonalidade no volume e na satisfação? Os picos de venda coincidem com quedas de satisfação?
- Dados: orders + reviews por mês
- Métrica: volume mensal + nota média mensal
- Impacto: se sim, operação precisa escalar em datas específicas

**Pergunta que NÃO conseguimos responder (Desafio):**
"Reduzir o prazo de entrega em X dias causaria aumento de Y pontos na satisfação?"
Isso exigiria inferência causal (experimento ou instrumental variable), não apenas correlação.

In [ ]:
# Salvar dados limpos
orders_clean.to_csv(f'{DATA_DIR}/orders_clean.csv', index=False)
items_clean.to_csv(f'{DATA_DIR}/items_clean.csv', index=False)
reviews_clean.to_csv(f'{DATA_DIR}/reviews_clean.csv', index=False)

print('Dados limpos salvos.')
print(f'  orders_clean:  {len(orders_clean):,}')
print(f'  items_clean:   {len(items_clean):,}')
print(f'  reviews_clean: {len(reviews_clean):,}')

if IN_COLAB:
    from google.colab import files
    print('\nBaixando arquivos limpos...')
    files.download(f'{DATA_DIR}/orders_clean.csv')
    files.download(f'{DATA_DIR}/items_clean.csv')
    files.download(f'{DATA_DIR}/reviews_clean.csv')